# NERISAL head detector: training notebook (Team Iconic)

Trains **YOLOv8n to detect heads** on CrowdHuman, in two sizes:
- **head640**: accurate tier (tiled counts)
- **head320**: fast watch tier (fine-tuned from head640)

**Before running:** Settings → Accelerator **GPU** (T4 or P100), Internet **On**, phone-verified account.

**Best way to run:** click **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 h,
even if you close the browser, and everything in `/kaggle/working` is saved as the notebook output.

Dataset: CrowdHuman (head boxes, `hbox`), licence CC BY-NC 4.0, non-commercial research and education only.
Do not redistribute the images.

In [ ]:
# 1. Settings: change these if you need a faster or slower run
TRAIN_ZIPS   = ["CrowdHuman_train01.zip", "CrowdHuman_train02.zip"]   # add "CrowdHuman_train03.zip" for more data
VAL_IMAGES   = 1000        # validation images used during training (more = slower epochs)
HOURS_640    = 3.5         # time budget for head640 (training stops at this limit or when it stops improving)
HOURS_320    = 1.5         # time budget for head320 (starts from head640's weights, so needs less)
BATCH_640    = 32
BATCH_320    = 64
BASE_MODEL   = "yolov8n.pt"  # or "yolo11n.pt" to try YOLO11

RAW  = "/tmp/crowdhuman"      # big temporary files (not saved)
DATA = "/tmp/heads"           # converted dataset (not saved)
OUT  = "/kaggle/working"      # saved: weights, plots, summary

In [ ]:
# 2. Environment check and install
import os, shutil, subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
for p in ("/tmp", "/kaggle/working"):
    t, u, f = shutil.disk_usage(p); print(f"{p}: {f/1e9:.0f} GB free")
!pip install -q ultralytics huggingface_hub
os.environ["WANDB_MODE"] = "disabled"          # W&B login prompt would freeze a background (commit) run
from ultralytics import settings
settings.update({k: False for k in ("wandb", "comet", "mlflow", "clearml", "neptune", "dvc") if k in settings})
import torch; print("torch", torch.__version__, "CUDA:", torch.cuda.is_available())

In [ ]:
# 3. Download CrowdHuman from Hugging Face (only the files we need)
from huggingface_hub import list_repo_files, hf_hub_download
REPO = "ShantyCam/crowdhuman"
TOKEN = None
try:                                   # only needed if the dataset asks you to log in / accept terms
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("using HF_TOKEN from Kaggle secrets")
except Exception:
    print("no HF_TOKEN secret (fine if the download works)")
files = list_repo_files(REPO, repo_type="dataset", token=TOKEN)
want = TRAIN_ZIPS + ["CrowdHuman_val.zip", "annotation_train.odgt", "annotation_val.odgt"]
os.makedirs(RAW, exist_ok=True)
paths = {}
for w in want:
    match = [f for f in files if f.split("/")[-1] == w]
    assert match, f"{w} not found in {REPO}. Files: {files[:20]}"
    print("downloading", match[0])
    paths[w] = hf_hub_download(REPO, match[0], repo_type="dataset", local_dir=RAW, token=TOKEN)
print("done"); print({k: f"{os.path.getsize(v)/1e9:.2f} GB" for k, v in paths.items()})

In [ ]:
# 4. Unzip images (each zip extracts an Images/ folder), then delete the zips to save disk
import zipfile
for z in TRAIN_ZIPS + ["CrowdHuman_val.zip"]:
    dest = os.path.join(RAW, "val" if "val" in z else "train")
    os.makedirs(dest, exist_ok=True)
    print("unzipping", z)
    with zipfile.ZipFile(paths[z]) as zf:
        zf.extractall(dest)
    os.remove(paths[z])
for split in ("train", "val"):
    n = sum(len(fs) for _, _, fs in os.walk(os.path.join(RAW, split)))
    print(split, n, "files")
t, u, f = shutil.disk_usage("/tmp"); print(f"/tmp: {f/1e9:.0f} GB free")

In [ ]:
# 5. Convert CrowdHuman head boxes to YOLO format
import json, glob, random
from PIL import Image

def find_images(root):
    idx = {}
    for p in glob.glob(os.path.join(root, "**", "*.jpg"), recursive=True):
        idx[os.path.splitext(os.path.basename(p))[0]] = p
    return idx

def convert(odgt, img_index, split, limit=None, seed=0):
    """Write DATA/images/<split>/<id>.jpg (symlink) and DATA/labels/<split>/<id>.txt.
    Keeps head boxes of real people only: skips 'mask' (ignore) regions and heads flagged
    ignore/unsure, clips boxes to the image, drops boxes smaller than 2 px."""
    lines = open(odgt).read().splitlines()
    random.Random(seed).shuffle(lines)
    os.makedirs(f"{DATA}/images/{split}", exist_ok=True)
    os.makedirs(f"{DATA}/labels/{split}", exist_ok=True)
    n_img = n_box = n_skip = 0
    for line in lines:
        if limit and n_img >= limit:
            break
        rec = json.loads(line)
        iid = rec["ID"]
        src = img_index.get(iid)
        if src is None:                       # image is in a zip we did not download
            continue
        W, H = Image.open(src).size
        rows = []
        for g in rec.get("gtboxes", []):
            if g.get("tag") != "person" or "hbox" not in g:
                n_skip += 1; continue
            ha = g.get("head_attr", {}) or {}
            if ha.get("ignore", 0) == 1 or ha.get("unsure", 0) == 1:
                n_skip += 1; continue
            x, y, w, h = g["hbox"]
            x1, y1 = max(0.0, x), max(0.0, y)
            x2, y2 = min(float(W), x + w), min(float(H), y + h)
            if x2 - x1 < 2 or y2 - y1 < 2:
                n_skip += 1; continue
            rows.append(f"0 {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
        dst = f"{DATA}/images/{split}/{iid}.jpg"
        if not os.path.exists(dst):
            try:
                os.link(src, dst)            # hard link: no extra disk, behaves like a normal file
            except OSError:
                shutil.copy(src, dst)
        with open(f"{DATA}/labels/{split}/{iid}.txt", "w") as f:
            f.write("\n".join(rows))
        n_img += 1; n_box += len(rows)
    print(f"{split}: {n_img} images, {n_box} heads ({n_box/max(n_img,1):.1f}/image), {n_skip} boxes skipped")
    return n_img

if os.path.exists(DATA):
    shutil.rmtree(DATA)
convert(paths["annotation_train.odgt"], find_images(os.path.join(RAW, "train")), "train")
convert(paths["annotation_val.odgt"],   find_images(os.path.join(RAW, "val")),   "val", limit=VAL_IMAGES)

yaml_path = f"{DATA}/heads.yaml"
open(yaml_path, "w").write(f"path: {DATA}\ntrain: images/train\nval: images/val\nnames:\n  0: head\n")
print(open(yaml_path).read())

In [ ]:
# 6. Sanity check: draw the head boxes on 3 random training images (look at them!)
import matplotlib.pyplot as plt, cv2
ims = random.Random(1).sample(sorted(glob.glob(f"{DATA}/images/train/*.jpg")), 3)
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
for ax, p in zip(axs, ims):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); H, W = im.shape[:2]
    lab = p.replace("/images/", "/labels/").replace(".jpg", ".txt")
    rows = [list(map(float, l.split()[1:])) for l in open(lab).read().splitlines() if l.strip()]
    for cx, cy, w, h in rows:
        x1, y1, x2, y2 = int((cx-w/2)*W), int((cy-h/2)*H), int((cx+w/2)*W), int((cy+h/2)*H)
        cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 0), max(1, W//400))
    ax.imshow(im); ax.set_title(f"{len(rows)} heads"); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/label_check.png", dpi=80); plt.show()

In [ ]:
# 7. Train head640 (accurate tier)
from ultralytics import YOLO
m640 = YOLO(BASE_MODEL)
m640.train(data=yaml_path, imgsz=640, epochs=100, time=HOURS_640, patience=15,
           batch=BATCH_640, device=0, workers=4, cache=False,
           project=f"{OUT}/runs", name="head640", exist_ok=True,
           close_mosaic=10, plots=True, verbose=False)
best640 = f"{OUT}/runs/head640/weights/best.pt"
print("best:", best640, os.path.exists(best640))

In [ ]:
# 8. Train head320 (fast watch tier), starting from head640's weights
m320 = YOLO(best640)
m320.train(data=yaml_path, imgsz=320, epochs=60, time=HOURS_320, patience=10,
           batch=BATCH_320, device=0, workers=4, cache=False,
           project=f"{OUT}/runs", name="head320", exist_ok=True,
           close_mosaic=5, plots=True, verbose=False)
best320 = f"{OUT}/runs/head320/weights/best.pt"
print("best:", best320, os.path.exists(best320))

In [ ]:
# 9. Counting check on held-out validation images: predicted head count vs true count
#    (this is the number that matters for NERISAL, not just mAP)
import numpy as np, csv
val_imgs = sorted(glob.glob(f"{DATA}/images/val/*.jpg"))[:300]
def true_count(p):
    lab = p.replace("/images/", "/labels/").replace(".jpg", ".txt")
    return sum(1 for l in open(lab) if l.strip())
rows = []
for name, w, sz in (("head640", best640, 640), ("head320", best320, 320)):
    if not os.path.exists(w): continue
    m = YOLO(w)
    for conf in (0.15, 0.25, 0.35):
        errs, pct = [], []
        for r in m.predict(val_imgs, imgsz=sz, conf=conf, max_det=1000, verbose=False, stream=True):
            t = true_count(r.path); n = len(r.boxes)
            errs.append(abs(n - t)); pct.append(abs(n - t) / max(t, 1) * 100)
        rows.append([name, conf, f"{np.mean(errs):.2f}", f"{np.sqrt(np.mean(np.square(errs))):.2f}", f"{np.median(pct):.1f}"])
        print(f"{name} conf={conf}: MAE {np.mean(errs):.2f}  RMSE {np.sqrt(np.mean(np.square(errs))):.2f}  median % error {np.median(pct):.1f}")
with open(f"{OUT}/count_check.csv", "w", newline="") as f:
    csv.writer(f).writerows([["model", "conf", "MAE", "RMSE", "median_pct_err"]] + rows)

In [ ]:
# 10. Package the weights for download (Output tab → nerisal_head_models.zip)
import zipfile
with zipfile.ZipFile(f"{OUT}/nerisal_head_models.zip", "w") as z:
    for n in ("head640", "head320"):
        for f in ("weights/best.pt", "results.csv", "results.png", "confusion_matrix.png"):
            p = f"{OUT}/runs/{n}/{f}"
            if os.path.exists(p):
                z.write(p, f"{n}/{os.path.basename(f) if f != 'weights/best.pt' else 'best.pt'}")
    for extra in ("count_check.csv", "label_check.png"):
        if os.path.exists(f"{OUT}/{extra}"): z.write(f"{OUT}/{extra}", extra)
print(zipfile.ZipFile(f"{OUT}/nerisal_head_models.zip").namelist())
print("Next: download the zip, then export best.pt files to .pte in Colab (see the export notebook).")